# 02. 문서 검색 (RAG)
FAISS와 HuggingFace Embedding을 이용한 검색 로직입니다.

In [ ]:
import os
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

VECTOR_DB_PATH = "../data/processed/faiss_index"
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"

def get_embeddings():
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

def init_or_load_vector_db() -> FAISS:
    embeddings = get_embeddings()
    if os.path.exists(VECTOR_DB_PATH):
        print("기존 DB 로드")
        return FAISS.load_local(VECTOR_DB_PATH, embeddings, allow_dangerous_deserialization=True)
    else:
        print("임시 DB 생성")
        dummy_docs = [
            Document(page_content="ACCURE Battery Intelligence는 SOH/RUL 예측 기업입니다.", metadata={"source_type": "company_website", "evidence_grade": "C", "is_direct": True})
        ]
        db = FAISS.from_documents(dummy_docs, embeddings)
        os.makedirs(os.path.dirname(VECTOR_DB_PATH), exist_ok=True)
        db.save_local(VECTOR_DB_PATH)
        return db

def retrieve_top_k(query: str, k: int = 3):
    db = init_or_load_vector_db()
    return db.similarity_search_with_score(query, k=k)
